In [1]:
# -*- coding: utf-8 -*-
"""
extraer_mfcc_secuencia.py
=========================
CTFG - Clasificacion de enfermedades respiratorias con deep learning.

Genera la representacion de entrada para las redes LSTM/GRU: una matriz de
13 MFCC en secuencia temporal por cada una de las 956 ventanas del subconjunto
balanceado del TFG.

Idea clave de reproducibilidad
------------------------------
El CSV del subconjunto (dataset_60_topado.csv) ya contiene, fila a fila, el
nombre del archivo de audio y el indice de la ventana DENTRO de ese archivo.
Ese indice es la posicion bruta del segmento de 5 s (indice 0 = muestras
[0:20000], indice 1 = [20000:40000], etc.), contando todos los segmentos
aunque algunos se descartaran despues por el filtro de silencio o por el tope
de 30 ventanas por paciente. Por tanto NO hay que volver a aplicar ni el filtro
de silencio ni el tope: basta con leer cada par (archivo, ventana) del CSV y
recortar exactamente ese segmento. Asi se calculan las MFCC sobre las mismas
956 ventanas que uso el TFG, con su mismo fold asignado.

Salida
------
Un fichero .npz comprimido con:
    X        -> float32, forma (956, MAX_FRAMES, 13). Ejes (muestra, tiempo, MFCC).
    y_str    -> etiquetas de clase como texto (ASTHMA, BRON, ...).
    y_int    -> etiquetas codificadas a entero, segun CLASES (orden fijo).
    pid      -> identificador de paciente (para trazabilidad).
    fold     -> fold de validacion 1..10 (mismos folds que el TFG).
    archivo  -> nombre del WAV de origen.
    ventana  -> indice de ventana dentro del archivo.
    clases   -> vector con el orden de clases usado para y_int.
Las MFCC se guardan SIN normalizar. La normalizacion (z-score por coeficiente)
debe hacerse dentro de cada fold, ajustada solo sobre el entrenamiento, igual
que el MinMaxScaler del TFG. Hacerla aqui introduciria fuga de informacion.

Notas para CPU
--------------
Este paso solo carga audio y calcula MFCC: es ligero (del orden de uno a tres
minutos para las 956 ventanas en un portatil sin GPU). El coste real esta en el
ENTRENAMIENTO de las RNN, no aqui. La longitud temporal elegida (MAX_FRAMES)
condiciona ese coste: 500 frames es fiel a la decision cerrada del CTFG, pero
duplica el tiempo de backpropagation a traves del tiempo frente a 250 frames.
Si el entrenamiento de LSTM/GRU resulta inviable en horas, la palanca mas
directa es bajar a HOP_LENGTH = 80 y MAX_FRAMES = 250 (ver bloque de config).
"""

import os
import sys
import json
import numpy as np
import pandas as pd
import librosa

# ===========================================================================
# CONFIGURACION  (lo unico que normalmente hay que tocar)
# ===========================================================================

# --- Rutas -----------------------------------------------------------------
# CSV del subconjunto balanceado del TFG (956 ventanas + folds).
CSV_SUBCONJUNTO = r"C:\Users\josem\Desktop\tfg\dataset_60_topado.csv"

# Carpeta(s) raiz donde estan los WAV originales (ICBHI + Fraiwan).
# El script busca de forma recursiva por nombre de archivo, asi que da igual
# como esten repartidos en subcarpetas. Puedes poner una o varias rutas.
WAV_ROOT_DIRS = [
    r"C:\Users\josem\Desktop\tfg",
]

# Fichero de salida con las MFCC en secuencia.
OUT_NPZ = r"C:\Users\josem\Desktop\tfg\mfcc_secuencia_60.npz"

# --- Parametros de audio (HEREDADOS del TFG, no cambiar) -------------------
SR = 4000                  # frecuencia de muestreo objetivo
WIN_SECONDS = 5            # duracion de la ventana
SAMPLES_PER_WINDOW = SR * WIN_SECONDS   # = 20000 muestras por ventana

# --- Parametros de la representacion MFCC en secuencia ---------------------
# Decision cerrada del CTFG: ~500 frames. Con hop=40 (10 ms) salen 501 frames,
# que se recortan a MAX_FRAMES=500.
N_MFCC = 13                # numero de coeficientes (mismo que el TFG)
N_FFT = 256                # tamano de FFT (4000/256 ~= 15.6 Hz por bin)
WIN_LENGTH = 100           # ventana de analisis = 25 ms a 4 kHz
HOP_LENGTH = 40            # salto = 10 ms a 4 kHz  -> ~500 frames en 5 s
N_MELS = 40                # filtros mel (40 evita filtros vacios a 4 kHz)
FMIN = 0
FMAX = 2000                # Nyquist a 4 kHz; el contenido pulmonar esta < 2 kHz
MAX_FRAMES = 500           # longitud temporal fija (recorta o rellena con ceros)

# --- ALTERNATIVA LIGERA PARA CPU (si el entrenamiento RNN no cabe en horas):
#       HOP_LENGTH = 80    -> 20 ms -> ~250 frames
#       MAX_FRAMES = 250
#     Reduce a la mitad los pasos temporales y, con ello, el tiempo de las RNN.

# --- Orden fijo de clases (define la codificacion entera y_int) ------------
CLASES = ["NORMAL", "ASTHMA", "BRON", "COPD", "HEART FAILURE", "PNEUMONIA"]

# ===========================================================================
# FUNCIONES AUXILIARES
# ===========================================================================

def construir_indice_wav(root_dirs):
    """Recorre las carpetas raiz y devuelve un dict {nombre_archivo: ruta}.
    Avisa si encuentra nombres de archivo duplicados en distintas carpetas."""
    indice = {}
    duplicados = []
    for root in root_dirs:
        if not os.path.isdir(root):
            print(f"  [AVISO] La carpeta no existe: {root}")
            continue
        for dirpath, _, files in os.walk(root):
            for f in files:
                if f.lower().endswith(".wav"):
                    if f in indice and indice[f] != os.path.join(dirpath, f):
                        duplicados.append(f)
                    indice[f] = os.path.join(dirpath, f)
    if duplicados:
        print(f"  [AVISO] {len(set(duplicados))} nombres de WAV duplicados en "
              f"distintas carpetas. Se usa la ultima ruta encontrada.")
    return indice


def mfcc_secuencia(y):
    """Calcula la matriz MFCC (tiempo x coeficiente) de una ventana de audio
    y la ajusta a longitud temporal fija MAX_FRAMES."""
    # librosa devuelve (n_mfcc, n_frames); transponemos a (n_frames, n_mfcc)
    m = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    ).T  # -> (n_frames, 13)

    n = m.shape[0]
    if n >= MAX_FRAMES:
        m = m[:MAX_FRAMES]                      # recorta los frames sobrantes
    else:
        pad = np.zeros((MAX_FRAMES - n, N_MFCC), dtype=m.dtype)
        m = np.vstack([m, pad])                 # rellena con ceros al final
    return m.astype(np.float32)


# ===========================================================================
# PROGRAMA PRINCIPAL
# ===========================================================================

def main():
    # 1) Cargar el CSV del subconjunto (separador ';', decimal ',') ----------
    if not os.path.isfile(CSV_SUBCONJUNTO):
        sys.exit(f"[ERROR] No encuentro el CSV del subconjunto: {CSV_SUBCONJUNTO}")
    df = pd.read_csv(CSV_SUBCONJUNTO, sep=";", decimal=",")
    print(f"CSV cargado: {len(df)} ventanas, {df['pid'].nunique()} pacientes, "
          f"{df['archivo'].nunique()} archivos.")

    # El indice de ventana viene como float (p.ej. 5,0); a entero.
    df["ventana"] = df["ventana"].round().astype(int)

    # 2) Indexar los WAV por nombre -----------------------------------------
    print("Indexando archivos WAV...")
    indice_wav = construir_indice_wav(WAV_ROOT_DIRS)
    print(f"  {len(indice_wav)} WAV encontrados en las carpetas raiz.")

    # Comprobar que estan todos los archivos del CSV antes de procesar.
    requeridos = set(df["archivo"].unique())
    faltan = sorted(a for a in requeridos if a not in indice_wav)
    if faltan:
        print(f"[ERROR] Faltan {len(faltan)} archivos WAV. Primeros ejemplos:")
        for a in faltan[:10]:
            print(f"        - {a}")
        sys.exit("Corrige WAV_ROOT_DIRS y vuelve a ejecutar.")

    # 3) Cache de senales por archivo (cada WAV se carga una sola vez) -------
    #    Varias ventanas comparten archivo; cargar una vez ahorra tiempo.
    cache_audio = {}

    def cargar(archivo):
        if archivo not in cache_audio:
            # mono=True por defecto; sr=4000 igual que el TFG.
            cache_audio[archivo], _ = librosa.load(indice_wav[archivo], sr=SR)
        return cache_audio[archivo]

    # 4) Recorrer filas del CSV y extraer la secuencia MFCC -----------------
    X = np.zeros((len(df), MAX_FRAMES, N_MFCC), dtype=np.float32)
    avisos_cortas = 0

    for i, fila in enumerate(df.itertuples(index=False)):
        senal = cargar(fila.archivo)
        ini = int(fila.ventana) * SAMPLES_PER_WINDOW
        fin = ini + SAMPLES_PER_WINDOW
        ventana_audio = senal[ini:fin]

        # Defensivo: las ventanas del CSV deberian ser completas (20000),
        # pero el remuestreo puede dejar la ultima ligeramente corta.
        if len(ventana_audio) < SAMPLES_PER_WINDOW:
            avisos_cortas += 1
            ventana_audio = np.pad(
                ventana_audio, (0, SAMPLES_PER_WINDOW - len(ventana_audio))
            )

        X[i] = mfcc_secuencia(ventana_audio)

        if (i + 1) % 100 == 0 or (i + 1) == len(df):
            print(f"  {i + 1:4d}/{len(df)} ventanas procesadas", end="\r")
    print()

    if avisos_cortas:
        print(f"  [AVISO] {avisos_cortas} ventanas venian incompletas y se "
              f"rellenaron con ceros (revisar si el numero es alto).")

    # 5) Etiquetas, codificacion y metadatos --------------------------------
    y_str = df["diagnostico"].to_numpy()
    clase_a_int = {c: k for k, c in enumerate(CLASES)}
    desconocidas = sorted(set(y_str) - set(CLASES))
    if desconocidas:
        sys.exit(f"[ERROR] Clases no previstas en CLASES: {desconocidas}")
    y_int = np.array([clase_a_int[c] for c in y_str], dtype=np.int64)

    pid = df["pid"].to_numpy()
    fold = df["fold"].to_numpy().astype(np.int64)
    archivo = df["archivo"].to_numpy()
    ventana = df["ventana"].to_numpy().astype(np.int64)

    # 6) Guardar -------------------------------------------------------------
    np.savez_compressed(
        OUT_NPZ,
        X=X, y_str=y_str, y_int=y_int,
        pid=pid, fold=fold, archivo=archivo, ventana=ventana,
        clases=np.array(CLASES),
    )

    # 7) Resumen final -------------------------------------------------------
    print("\n=== RESUMEN ===")
    print(f"X: forma {X.shape}, dtype {X.dtype} "
          f"({X.nbytes / 1e6:.1f} MB en memoria)")
    print(f"Frames por ventana: {MAX_FRAMES} | coeficientes: {N_MFCC}")
    print("Ventanas por clase:")
    for c in CLASES:
        print(f"  {c:14s}: {(y_str == c).sum()}")
    print("Ventanas por fold:")
    for f in range(1, 11):
        print(f"  fold {f:2d}: {(fold == f).sum()}")
    print(f"\nGuardado en: {OUT_NPZ}")

    # Manifiesto JSON con los parametros usados (para citar en la memoria).
    manifiesto = {
        "sr": SR, "win_seconds": WIN_SECONDS,
        "samples_per_window": SAMPLES_PER_WINDOW,
        "n_mfcc": N_MFCC, "n_fft": N_FFT, "win_length": WIN_LENGTH,
        "hop_length": HOP_LENGTH, "n_mels": N_MELS,
        "fmin": FMIN, "fmax": FMAX, "max_frames": MAX_FRAMES,
        "n_ventanas": int(len(df)), "clases": CLASES,
    }
    with open(os.path.splitext(OUT_NPZ)[0] + "_manifiesto.json", "w",
              encoding="utf-8") as fh:
        json.dump(manifiesto, fh, indent=2, ensure_ascii=False)
    print("Manifiesto de parametros guardado junto al .npz.")


if __name__ == "__main__":
    main()

CSV cargado: 956 ventanas, 60 pacientes, 263 archivos.
Indexando archivos WAV...
  1256 WAV encontrados en las carpetas raiz.
   956/956 ventanas procesadas

=== RESUMEN ===
X: forma (956, 500, 13), dtype float32 (24.9 MB en memoria)
Frames por ventana: 500 | coeficientes: 13
Ventanas por clase:
  NORMAL        : 136
  ASTHMA        : 135
  BRON          : 106
  COPD          : 300
  HEART FAILURE : 111
  PNEUMONIA     : 168
Ventanas por fold:
  fold  1: 133
  fold  2: 117
  fold  3: 107
  fold  4: 96
  fold  5: 94
  fold  6: 86
  fold  7: 83
  fold  8: 83
  fold  9: 83
  fold 10: 74

Guardado en: C:\Users\josem\Desktop\tfg\mfcc_secuencia_60.npz
Manifiesto de parametros guardado junto al .npz.
